# Лабораторная работа №2 - ETL реализованный с помощью Spark

## 1. Настройка SparkSession и подключений

In [64]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import psycopg2

spark = (SparkSession.builder
    .appName("Lab2_ETL")
    .config("spark.jars", "/home/jovyan/jars/postgresql-42.7.3.jar,/home/jovyan/jars/clickhouse-jdbc-0.6.0-all.jar")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate())

PG_URL   = "jdbc:postgresql://postgres:5432/postgres"
PG_PROPS = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

CH_URL   = "jdbc:clickhouse://clickhouse:8123/lab2"
CH_PROPS = {"user": "admin", "password": "admin123", "driver": "com.clickhouse.jdbc.ClickHouseDriver"}

UNKNOWN = "__UNKNOWN__"


def read_pg(table):
    return spark.read.jdbc(PG_URL, table, properties=PG_PROPS)

def write_pg(df, table, mode="overwrite"):
    df.write.jdbc(PG_URL, table, mode=mode, properties=PG_PROPS)

def write_ch(df, table, order_by):
    (df.write
        .format("jdbc")
        .option("url", CH_URL)
        .option("dbtable", table)
        .option("user", CH_PROPS["user"])
        .option("password", CH_PROPS["password"])
        .option("driver", CH_PROPS["driver"])
        .option("createTableOptions", f"ENGINE = MergeTree() ORDER BY ({order_by})")
        .mode("overwrite")
        .save())

def pg_execute(sql):
    conn = psycopg2.connect(host="postgres", port=5432, dbname="postgres",
                            user="postgres", password="postgres")
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute(sql)
    conn.close()

def fill_unknown(col_name, alias=None):
    name = alias if alias else col_name
    return (
        F.when(F.col(col_name).isNull() | (F.trim(F.col(col_name)) == ""), F.lit(UNKNOWN))
         .otherwise(F.trim(F.col(col_name)))
         .alias(name)
    )


print("Spark", spark.version)

Spark 3.5.0


## 2. Чтение исходных данных (mock_data)

In [65]:
mock_data = read_pg("mock_data")
mock_data.cache()

row_count = mock_data.count()
assert row_count == 10_000, f"Ожидалось 10 000 строк, получено {row_count}"
print(f"mock_data: {row_count} строк")
mock_data.printSchema()

mock_data: 10000 строк
root
 |-- id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_customer_id: int

## 3. Трансформация в схему «звезда»

### 3.1 Вспомогательные функции (PK/FK)

In [66]:
def add_pk(table, column):
    pk_name = f"{table}_pkey"
    pg_execute(f'ALTER TABLE "{table}" ALTER COLUMN "{column}" SET NOT NULL;')
    pg_execute(f"""
        DO $$ BEGIN
            IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = '{pk_name}') THEN
                ALTER TABLE "{table}" ADD CONSTRAINT "{pk_name}" PRIMARY KEY ("{column}");
            END IF;
        END $$;
    """)
    print(f"  PK ready: {table}.{column}")


def add_fk(src, name, src_col, tgt, tgt_col):
    pg_execute(f"""
        DO $$ BEGIN
            IF to_regclass('public.{src}') IS NOT NULL
               AND to_regclass('public.{tgt}') IS NOT NULL
               AND NOT EXISTS (
                   SELECT 1 FROM pg_constraint c
                   JOIN pg_class t ON t.oid = c.conrelid
                   WHERE c.conname = '{name}' AND t.relname = '{src}'
               )
            THEN
                ALTER TABLE "{src}"
                ADD CONSTRAINT "{name}"
                FOREIGN KEY ("{src_col}") REFERENCES "{tgt}"("{tgt_col}");
            END IF;
        END $$;
    """)


def drop_fks(table):
    pg_execute(f"""
        DO $$ DECLARE r record; BEGIN
            IF to_regclass('public.{table}') IS NOT NULL THEN
                FOR r IN
                    SELECT c.conname FROM pg_constraint c
                    JOIN pg_class t ON t.oid = c.conrelid
                    WHERE c.contype = 'f' AND t.relname = '{table}'
                LOOP
                    EXECUTE format('ALTER TABLE %I DROP CONSTRAINT IF EXISTS %I', '{table}', r.conname);
                END LOOP;
            END IF;
        END $$;
    """)

print("Helpers ready")

Helpers ready


### 3.2 Сброс состояния (для повторного запуска)

In [67]:
drop_fks("fact_sales")
print("FK constraints dropped")

FK constraints dropped


### 3.3 dim_customer — клиенты

PK: `customer_id` — натуральный ключ из CSV (с offset). Включает данные питомца клиента.

In [68]:
dim_customer = (
    mock_data.select(
        F.col("sale_customer_id").alias("customer_id"),
        F.trim(F.col("customer_first_name")).alias("customer_first_name"),
        F.trim(F.col("customer_last_name")).alias("customer_last_name"),
        F.col("customer_age").alias("customer_age"),
        F.trim(F.col("customer_email")).alias("customer_email"),
        F.trim(F.col("customer_country")).alias("customer_country"),
        F.trim(F.col("customer_postal_code")).alias("customer_postal_code"),
        F.trim(F.col("customer_pet_type")).alias("pet_type"),
        F.trim(F.col("customer_pet_name")).alias("pet_name"),
        F.trim(F.col("customer_pet_breed")).alias("pet_breed"),
        F.trim(F.col("pet_category")).alias("pet_category"),
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

write_pg(dim_customer, "dim_customer")
print(f"dim_customer: {dim_customer.count()} rows")

dim_customer: 10000 rows


### 3.4 dim_seller — продавцы

PK: `seller_id` — натуральный ключ из CSV (с offset).

In [69]:
dim_seller = (
    mock_data.select(
        F.col("sale_seller_id").alias("seller_id"),
        F.trim(F.col("seller_first_name")).alias("seller_first_name"),
        F.trim(F.col("seller_last_name")).alias("seller_last_name"),
        F.trim(F.col("seller_email")).alias("seller_email"),
        F.trim(F.col("seller_country")).alias("seller_country"),
        F.trim(F.col("seller_postal_code")).alias("seller_postal_code"),
    )
    .filter(F.col("seller_id").isNotNull())
    .dropDuplicates(["seller_id"])
)

write_pg(dim_seller, "dim_seller")
print(f"dim_seller: {dim_seller.count()} rows")

dim_seller: 10000 rows


### 3.5 dim_product — товары

PK: `product_id` — натуральный ключ из CSV (с offset). Даты парсятся из формата `M/d/yyyy`.

In [70]:
dim_product = (
    mock_data.select(
        F.col("sale_product_id").alias("product_id"),
        F.trim(F.col("product_name")).alias("product_name"),
        F.trim(F.col("product_category")).alias("product_category"),
        F.col("product_price"),
        F.col("product_quantity"),
        F.col("product_weight"),
        F.trim(F.col("product_color")).alias("product_color"),
        F.trim(F.col("product_size")).alias("product_size"),
        F.trim(F.col("product_brand")).alias("product_brand"),
        F.trim(F.col("product_material")).alias("product_material"),
        F.trim(F.col("product_description")).alias("product_description"),
        F.col("product_rating"),
        F.col("product_reviews"),
        F.when(
            F.col("product_release_date").rlike(r"^[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}$"),
            F.to_date(F.col("product_release_date"), "M/d/yyyy")
        ).otherwise(F.lit(None)).alias("product_release_date"),
        F.when(
            F.col("product_expiry_date").rlike(r"^[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}$"),
            F.to_date(F.col("product_expiry_date"), "M/d/yyyy")
        ).otherwise(F.lit(None)).alias("product_expiry_date"),
    )
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

write_pg(dim_product, "dim_product")
print(f"dim_product: {dim_product.count()} rows")

dim_product: 10000 rows


### 3.6 dim_store — магазины

PK: `store_id` — суррогатный ключ. Один магазин может встречаться в нескольких строках CSV,
поэтому сначала делаем `dropDuplicates` по всем атрибутам, затем генерируем `row_number`.

In [71]:
store_cols = ["store_name", "store_location", "store_city",
              "store_state", "store_country", "store_phone", "store_email"]

store_unique = (
    mock_data
    .select(*[fill_unknown(c) for c in store_cols])
    .dropDuplicates(store_cols)
)

w_store = Window.orderBy(*store_cols)
dim_store = store_unique.withColumn("store_id", F.row_number().over(w_store))
dim_store = dim_store.select("store_id", *store_cols)

write_pg(dim_store, "dim_store")

dim_store_db = read_pg("dim_store")
print(f"dim_store: {dim_store_db.count()} rows")

dim_store: 10000 rows


### 3.7 dim_supplier — поставщики

PK: `supplier_id` — суррогатный ключ. Аналогичная логика: дедупликация → `row_number`.

In [72]:
supplier_cols = ["supplier_name", "supplier_contact", "supplier_email",
                 "supplier_phone", "supplier_address", "supplier_city", "supplier_country"]

supplier_unique = (
    mock_data
    .select(*[fill_unknown(c) for c in supplier_cols])
    .dropDuplicates(supplier_cols)
)

w_supplier = Window.orderBy(*supplier_cols)
dim_supplier = supplier_unique.withColumn("supplier_id", F.row_number().over(w_supplier))
dim_supplier = dim_supplier.select("supplier_id", *supplier_cols)

write_pg(dim_supplier, "dim_supplier")

dim_supplier_db = read_pg("dim_supplier")
print(f"dim_supplier: {dim_supplier_db.count()} rows")

dim_supplier: 10000 rows


### 3.8 Первичные ключи dimension-таблиц

In [73]:
print("Adding primary keys...")
add_pk("dim_customer", "customer_id")
add_pk("dim_seller",   "seller_id")
add_pk("dim_product",  "product_id")
add_pk("dim_store",    "store_id")
add_pk("dim_supplier", "supplier_id")
print("All dim PKs created")

Adding primary keys...
  PK ready: dim_customer.customer_id
  PK ready: dim_seller.seller_id
  PK ready: dim_product.product_id
  PK ready: dim_store.store_id
  PK ready: dim_supplier.supplier_id
All dim PKs created


### 3.9 fact_sales — таблица фактов

Для магазина и поставщика нужен JOIN с уже записанными dim-таблицами, чтобы
сопоставить атрибуты строки с суррогатным ключом (`store_id`, `supplier_id`).
Для клиента, продавца, продукта — ID берутся напрямую из CSV (они уже уникальны после offset).

In [74]:
sales_base = mock_data.select(
    F.col("sale_customer_id"),
    F.col("sale_seller_id"),
    F.col("sale_product_id"),
    F.col("sale_quantity"),
    F.col("sale_total_price"),
    F.col("sale_date"),
    *[fill_unknown(c) for c in store_cols],
    *[fill_unknown(c) for c in supplier_cols],
)

fact_sales = (
    sales_base
    .join(dim_store_db.select("store_id", *store_cols), store_cols, "left")
    .join(dim_supplier_db.select("supplier_id", *supplier_cols), supplier_cols, "left")
    .select(
        F.to_date(F.col("sale_date"), "M/d/yyyy").alias("sale_date"),
        F.col("sale_customer_id").alias("customer_id"),
        F.col("sale_seller_id").alias("seller_id"),
        F.col("sale_product_id").alias("product_id"),
        F.col("store_id"),
        F.col("supplier_id"),
        F.col("sale_quantity"),
        F.col("sale_total_price"),
    )
)

w_sale = Window.orderBy("sale_date", "customer_id", "product_id")
fact_sales = (fact_sales
    .withColumn("sale_id", F.row_number().over(w_sale))
    .select("sale_id", "sale_date", "customer_id", "seller_id",
            "product_id", "store_id", "supplier_id",
            "sale_quantity", "sale_total_price")
)

write_pg(fact_sales, "fact_sales")
add_pk("fact_sales", "sale_id")

null_fks = fact_sales.filter(
    F.col("store_id").isNull() | F.col("supplier_id").isNull()
).count()
assert null_fks == 0, f"NULL в FK-полях fact_sales: {null_fks} строк"

print(f"fact_sales: {fact_sales.count()} строк, NULL в FK: 0")

  PK ready: fact_sales.sale_id
fact_sales: 10000 строк, NULL в FK: 0


### 3.10 Внешние ключи fact_sales → dim_*

In [75]:
print("Adding foreign keys...")
add_fk("fact_sales", "fk_sales_customer", "customer_id", "dim_customer", "customer_id")
add_fk("fact_sales", "fk_sales_seller",   "seller_id",   "dim_seller",   "seller_id")
add_fk("fact_sales", "fk_sales_product",  "product_id",  "dim_product",  "product_id")
add_fk("fact_sales", "fk_sales_store",    "store_id",    "dim_store",    "store_id")
add_fk("fact_sales", "fk_sales_supplier", "supplier_id", "dim_supplier", "supplier_id")
print("Star schema complete!")

Adding foreign keys...
Star schema complete!


---

## 4. Аналитические витрины

Читаем схему «звезда» из PostgreSQL, строим агрегации в PySpark и записываем 6 отчётов в ClickHouse.

In [76]:
fs  = read_pg("fact_sales").alias("fs")
dp  = read_pg("dim_product").alias("dp")
dcu = read_pg("dim_customer").alias("dcu")
dst = read_pg("dim_store").alias("dst")
dsu = read_pg("dim_supplier").alias("dsu")

print("Tables loaded for reports")

Tables loaded for reports


### 4.1 Витрина продаж по продуктам (`report_products`)

- Топ-10 самых продаваемых продуктов
- Выручка по категориям
- Средний рейтинг и количество отзывов

In [77]:
report_products = (
    fs
    .join(dp, F.col("fs.product_id") == F.col("dp.product_id"), "left")
    .groupBy(
        F.col("fs.product_id"),
        F.col("dp.product_name"),
        F.col("dp.product_category"),
    )
    .agg(
        F.sum("fs.sale_total_price").alias("total_revenue"),
        F.sum("fs.sale_quantity").alias("total_sales_quantity"),
        F.count(F.col("fs.sale_id")).alias("sales_count"),
        F.avg("dp.product_rating").alias("avg_product_rating"),
        F.max("dp.product_reviews").alias("product_reviews"),
    )
)

w_product = Window.orderBy(F.col("total_sales_quantity").desc(), F.col("total_revenue").desc())
report_products = (
    report_products
    .withColumn("product_rank_by_sales", F.row_number().over(w_product))
    .withColumn("is_top_10_product",
                F.when(F.col("product_rank_by_sales") <= 10, F.lit(1)).otherwise(F.lit(0)))
)

write_ch(report_products, "report_products", "product_id")
print(f"report_products: {report_products.count()} rows")
report_products.filter(F.col("is_top_10_product") == 1).show(10, truncate=False)

report_products: 10000 rows
+----------+------------+----------------+-------------+--------------------+-----------+------------------+---------------+---------------------+-----------------+
|product_id|product_name|product_category|total_revenue|total_sales_quantity|sales_count|avg_product_rating|product_reviews|product_rank_by_sales|is_top_10_product|
+----------+------------+----------------+-------------+--------------------+-----------+------------------+---------------+---------------------+-----------------+
|8616      |Cat Toy     |Toy             |499.73       |10                  |1          |1.600000          |290            |1                    |1                |
|3550      |Dog Food    |Toy             |499.59       |10                  |1          |1.900000          |769            |2                    |1                |
|1484      |Bird Cage   |Toy             |498.96       |10                  |1          |3.300000          |842            |3                    |1

### 4.2 Витрина продаж по клиентам (`report_customers`)

- Топ-10 клиентов по сумме покупок
- Распределение по странам
- Средний чек

In [78]:
country_dist = (
    dcu.groupBy("customer_country")
    .agg(F.countDistinct("customer_id").alias("customers_in_country"))
    .alias("cd")
)

report_customers = (
    fs
    .join(dcu, F.col("fs.customer_id") == F.col("dcu.customer_id"), "left")
    .groupBy(
        F.col("fs.customer_id"),
        F.col("dcu.customer_first_name"),
        F.col("dcu.customer_last_name"),
        F.col("dcu.customer_country"),
    )
    .agg(
        F.sum("fs.sale_total_price").alias("total_purchase_amount"),
        F.count(F.col("fs.sale_id")).alias("orders_count"),
        F.avg("fs.sale_total_price").alias("avg_check"),
    )
    .join(country_dist, F.col("dcu.customer_country") == F.col("cd.customer_country"), "left")
    .drop(F.col("cd.customer_country"))
)

w_customer = Window.orderBy(F.col("total_purchase_amount").desc())
report_customers = (
    report_customers
    .withColumn("customer_rank_by_revenue", F.row_number().over(w_customer))
    .withColumn("is_top_10_customer",
                F.when(F.col("customer_rank_by_revenue") <= 10, F.lit(1)).otherwise(F.lit(0)))
)

write_ch(report_customers, "report_customers", "customer_id")
print(f"report_customers: {report_customers.count()} rows")
report_customers.filter(F.col("is_top_10_customer") == 1).show(10, truncate=False)

report_customers: 10000 rows
+-----------+-------------------+------------------+----------------+---------------------+------------+----------+--------------------+------------------------+------------------+
|customer_id|customer_first_name|customer_last_name|customer_country|total_purchase_amount|orders_count|avg_check |customers_in_country|customer_rank_by_revenue|is_top_10_customer|
+-----------+-------------------+------------------+----------------+---------------------+------------+----------+--------------------+------------------------+------------------+
|4188       |Gus                |Hartshorn         |Albania         |499.85               |1           |499.850000|46                  |1                       |1                 |
|6422       |Hayes              |McKain            |Portugal        |499.80               |1           |499.800000|336                 |2                       |1                 |
|6361       |Ava                |Lomas             |China         

### 4.3 Витрина продаж по времени (`report_time`)

- Месячные тренды выручки и количества заказов
- MoM (месяц к месяцу): разница выручки с предыдущим месяцем
- YoY (год к году): данные только за 2021 год, поэтому `prev_year_same_month_revenue = 0`

In [79]:
report_time = (
    fs
    .withColumn("sale_year", F.year(F.col("fs.sale_date")))
    .withColumn("sale_month", F.month(F.col("fs.sale_date")))
    .groupBy("sale_year", "sale_month")
    .agg(
        F.sum("fs.sale_total_price").alias("monthly_revenue"),
        F.sum("fs.sale_quantity").alias("monthly_sales_quantity"),
        F.count(F.col("fs.sale_id")).alias("orders_count"),
        F.avg("fs.sale_total_price").alias("avg_order_size"),
    )
)

w_time = Window.orderBy("sale_year", "sale_month")
report_time = (
    report_time
    .withColumn("prev_month_revenue",
                F.coalesce(F.lag("monthly_revenue", 1).over(w_time), F.lit(0)))
    .withColumn("prev_year_same_month_revenue",
                F.coalesce(F.lag("monthly_revenue", 12).over(w_time), F.lit(0)))
    .withColumn("revenue_delta_vs_prev_month",
                F.col("monthly_revenue") - F.col("prev_month_revenue"))
    .withColumn("revenue_delta_vs_prev_year",
                F.col("monthly_revenue") - F.col("prev_year_same_month_revenue"))
)

write_ch(report_time, "report_time", "(sale_year, sale_month)")
print(f"report_time: {report_time.count()} rows")
report_time.orderBy("sale_year", "sale_month").show(30, truncate=False)

report_time: 12 rows
+---------+----------+---------------+----------------------+------------+--------------+------------------+----------------------------+---------------------------+--------------------------+
|sale_year|sale_month|monthly_revenue|monthly_sales_quantity|orders_count|avg_order_size|prev_month_revenue|prev_year_same_month_revenue|revenue_delta_vs_prev_month|revenue_delta_vs_prev_year|
+---------+----------+---------------+----------------------+------------+--------------+------------------+----------------------------+---------------------------+--------------------------+
|2021     |1         |224158.54      |4856                  |874         |256.474302    |0.00              |0.00                        |224158.54                  |224158.54                 |
|2021     |2         |192348.31      |4070                  |739         |260.281881    |224158.54         |0.00                        |-31810.23                  |192348.31                 |
|2021     |3  

### 4.4 Витрина продаж по магазинам (`report_stores`)

- Топ-5 магазинов по выручке
- Распределение по городам и странам
- Средний чек

In [80]:
report_stores = (
    fs
    .join(dst, F.col("fs.store_id") == F.col("dst.store_id"), "left")
    .groupBy(
        F.col("fs.store_id"),
        F.col("dst.store_name"),
        F.col("dst.store_city"),
        F.col("dst.store_country"),
    )
    .agg(
        F.sum("fs.sale_total_price").alias("total_revenue"),
        F.count(F.col("fs.sale_id")).alias("orders_count"),
        F.avg("fs.sale_total_price").alias("avg_check"),
    )
)

w_store_global  = Window.orderBy(F.col("total_revenue").desc())
w_store_city    = Window.partitionBy("store_city")
w_store_country = Window.partitionBy("store_country")

report_stores = (
    report_stores
    .withColumn("store_rank_by_revenue", F.row_number().over(w_store_global))
    .withColumn("is_top_5_store",
                F.when(F.col("store_rank_by_revenue") <= 5, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("city_total_revenue", F.sum("total_revenue").over(w_store_city))
    .withColumn("country_total_revenue", F.sum("total_revenue").over(w_store_country))
)

write_ch(report_stores, "report_stores", "store_id")
print(f"report_stores: {report_stores.count()} rows")
report_stores.filter(F.col("is_top_5_store") == 1).show(5, truncate=False)

report_stores: 10000 rows
+--------+-----------+----------+-------------+-------------+------------+----------+---------------------+--------------+------------------+---------------------+
|store_id|store_name |store_city|store_country|total_revenue|orders_count|avg_check |store_rank_by_revenue|is_top_5_store|city_total_revenue|country_total_revenue|
+--------+-----------+----------+-------------+-------------+------------+----------+---------------------+--------------+------------------+---------------------+
|2165    |Edgeblab   |Pesek     |Indonesia    |499.76       |1           |499.760000|4                    |1             |499.76            |279350.16            |
|7806    |Thoughtblab|Fonte     |Poland       |499.80       |1           |499.800000|2                    |1             |499.80            |81313.13             |
|1266    |Centizu    |Tylicz    |Poland       |499.73       |1           |499.730000|5                    |1             |641.52            |81313.13     

### 4.5 Витрина продаж по поставщикам (`report_suppliers`)

- Топ-5 поставщиков по выручке
- Средняя цена товаров
- Распределение по странам

In [81]:
report_suppliers = (
    fs
    .join(dsu, F.col("fs.supplier_id") == F.col("dsu.supplier_id"), "left")
    .join(dp,  F.col("fs.product_id")  == F.col("dp.product_id"),  "left")
    .groupBy(
        F.col("fs.supplier_id"),
        F.col("dsu.supplier_name"),
        F.col("dsu.supplier_country"),
    )
    .agg(
        F.sum("fs.sale_total_price").alias("total_revenue"),
        F.count(F.col("fs.sale_id")).alias("orders_count"),
        F.avg("dp.product_price").alias("avg_product_price"),
    )
)

w_supplier_global  = Window.orderBy(F.col("total_revenue").desc())
w_supplier_country = Window.partitionBy("supplier_country")

report_suppliers = (
    report_suppliers
    .withColumn("supplier_rank_by_revenue", F.row_number().over(w_supplier_global))
    .withColumn("is_top_5_supplier",
                F.when(F.col("supplier_rank_by_revenue") <= 5, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("country_total_revenue", F.sum("total_revenue").over(w_supplier_country))
)

write_ch(report_suppliers, "report_suppliers", "supplier_id")
print(f"report_suppliers: {report_suppliers.count()} rows")
report_suppliers.filter(F.col("is_top_5_supplier") == 1).show(5, truncate=False)

report_suppliers: 10000 rows
+-----------+-------------+----------------+-------------+------------+-----------------+------------------------+-----------------+---------------------+
|supplier_id|supplier_name|supplier_country|total_revenue|orders_count|avg_product_price|supplier_rank_by_revenue|is_top_5_supplier|country_total_revenue|
+-----------+-------------+----------------+-------------+------------+-----------------+------------------------+-----------------+---------------------+
|912        |Browsezoom   |Argentina       |499.73       |1           |58.240000        |5                       |1                |35606.32             |
|1528       |Demimbu      |China           |499.76       |1           |53.800000        |3                       |1                |492823.31            |
|710        |Brainverse   |Ireland         |499.85       |1           |31.420000        |1                       |1                |11636.18             |
|1948       |Eabox        |Portugal      

### 4.6 Витрина качества продукции (`report_quality`)

- Продукты с наивысшим/наименьшим рейтингом
- Корреляция рейтинга и объёма продаж
- Продукты с наибольшим количеством отзывов

In [82]:
report_quality = (
    fs
    .join(dp, F.col("fs.product_id") == F.col("dp.product_id"), "left")
    .groupBy(F.col("fs.product_id"), F.col("dp.product_name"))
    .agg(
        F.sum("fs.sale_quantity").alias("total_sales_quantity"),
        F.sum("fs.sale_total_price").alias("total_revenue"),
        F.avg("dp.product_rating").alias("avg_product_rating"),
        F.max("dp.product_reviews").alias("product_reviews"),
    )
)

w_rating_desc = Window.orderBy(F.col("avg_product_rating").desc(), F.col("product_reviews").desc())
w_rating_asc  = Window.orderBy(F.col("avg_product_rating").asc(), F.col("product_reviews").desc())
w_reviews     = Window.orderBy(F.col("product_reviews").desc())

report_quality = (
    report_quality
    .withColumn("highest_rating_rank", F.row_number().over(w_rating_desc))
    .withColumn("lowest_rating_rank",  F.row_number().over(w_rating_asc))
    .withColumn("reviews_rank",        F.row_number().over(w_reviews))
    .withColumn("is_highest_rated_product",
                F.when(F.col("highest_rating_rank") == 1, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_lowest_rated_product",
                F.when(F.col("lowest_rating_rank") == 1, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_most_reviewed_product",
                F.when(F.col("reviews_rank") == 1, F.lit(1)).otherwise(F.lit(0)))
)

corr_df = report_quality.agg(
    F.corr("avg_product_rating", "total_sales_quantity").alias("rating_sales_correlation")
)
report_quality = report_quality.crossJoin(corr_df)

write_ch(report_quality, "report_quality", "product_id")
print(f"report_quality: {report_quality.count()} rows")
report_quality.filter(
    (F.col("is_highest_rated_product") == 1) |
    (F.col("is_lowest_rated_product") == 1) |
    (F.col("is_most_reviewed_product") == 1)
).show(truncate=False)

report_quality: 10000 rows
+----------+------------+--------------------+-------------+------------------+---------------+-------------------+------------------+------------+------------------------+-----------------------+------------------------+------------------------+
|product_id|product_name|total_sales_quantity|total_revenue|avg_product_rating|product_reviews|highest_rating_rank|lowest_rating_rank|reviews_rank|is_highest_rated_product|is_lowest_rated_product|is_most_reviewed_product|rating_sales_correlation|
+----------+------------+--------------------+-------------+------------------+---------------+-------------------+------------------+------------+------------------------+-----------------------+------------------------+------------------------+
|2069      |Bird Cage   |7                   |256.17       |1.600000          |1000           |8374               |1382              |1           |0                       |0                      |1                       |0.001004977

---

## 5. Верификация результатов

Проверяем все контрольные точки из условия

In [83]:
errors = []

print("=== PostgreSQL ===")

pg_checks = [
    ("mock_data",    10_000),
    ("dim_customer", 10_000),
    ("dim_seller",   10_000),
    ("dim_product",  10_000),
    ("fact_sales",   10_000),
]
for tbl, expected in pg_checks:
    cnt = read_pg(tbl).count()
    ok = cnt == expected
    print(f"  {'OK' if ok else 'FAIL'} {tbl}: {cnt} (ожидалось {expected})")
    if not ok:
        errors.append(f"{tbl}: {cnt} != {expected}")

for tbl in ("dim_store", "dim_supplier"):
    cnt = read_pg(tbl).count()
    ok = 1 <= cnt <= 10_000
    print(f"  {'OK' if ok else 'FAIL'} {tbl}: {cnt} уникальных")
    if not ok:
        errors.append(f"{tbl}: {cnt}")

fs_check = read_pg("fact_sales")
fk_cols = ["customer_id", "seller_id", "product_id", "store_id", "supplier_id"]
for col in fk_cols:
    nulls = fs_check.filter(F.col(col).isNull()).count()
    print(f"  {'OK' if nulls == 0 else 'FAIL'} fact_sales.{col} NULL: {nulls}")
    if nulls > 0:
        errors.append(f"fact_sales.{col} — {nulls} NULL")

print("\n=== ClickHouse ===")

ch_tables = {
    "report_products":  (1, 10_000),
    "report_customers": (1, 10_000),
    "report_time":      12,
    "report_stores":    (1, 10_000),
    "report_suppliers": (1, 10_000),
    "report_quality":   (1, 10_000),
}
for tbl, expected in ch_tables.items():
    cnt = (spark.read.format("jdbc")
           .option("url", CH_URL)
           .option("dbtable", tbl)
           .option("user", CH_PROPS["user"])
           .option("password", CH_PROPS["password"])
           .option("driver", CH_PROPS["driver"])
           .load().count())
    if isinstance(expected, tuple):
        ok = expected[0] <= cnt <= expected[1]
    else:
        ok = cnt == expected
    print(f"  {'OK' if ok else 'FAIL'} {tbl}: {cnt}")
    if not ok:
        errors.append(f"{tbl}: {cnt}")

print()
if errors:
    print("FAIL — обнаружены проблемы:")
    for e in errors:
        print(f"  - {e}")
else:
    print("OK — все проверки пройдены. ETL-пайплайн выполнен корректно.")

=== PostgreSQL ===
  OK mock_data: 10000 (ожидалось 10000)
  OK dim_customer: 10000 (ожидалось 10000)
  OK dim_seller: 10000 (ожидалось 10000)
  OK dim_product: 10000 (ожидалось 10000)
  OK fact_sales: 10000 (ожидалось 10000)
  OK dim_store: 10000 уникальных
  OK dim_supplier: 10000 уникальных
  OK fact_sales.customer_id NULL: 0
  OK fact_sales.seller_id NULL: 0
  OK fact_sales.product_id NULL: 0
  OK fact_sales.store_id NULL: 0
  OK fact_sales.supplier_id NULL: 0

=== ClickHouse ===
  OK report_products: 10000
  OK report_customers: 10000
  OK report_time: 12
  OK report_stores: 10000
  OK report_suppliers: 10000
  OK report_quality: 10000

OK — все проверки пройдены. ETL-пайплайн выполнен корректно.


In [84]:
spark.stop()
print("Spark session stopped")

Spark session stopped
